In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("UrbanFlow - Data Quality") \
    .getOrCreate()

RAW_PATH = "gs://trip-data-project-e6d9c504-e176-49d1-acd/urbanflow/raw/hvfhv/"

df = spark.read.parquet(RAW_PATH)

print("Registros RAW:", f"{df.count():,}")
print("Columnas RAW:", len(df.columns))

df_quality = (
    df

    # Duración del viaje
    .withColumn(
        "trip_duration_minutes",
        F.col("trip_time") / 60.0
    )

    # Tiempo de espera válido
    .withColumn(
        "wait_time_minutes",
        F.when(
            F.col("request_datetime") <= F.col("pickup_datetime"),
            (
                F.unix_timestamp("pickup_datetime") -
                F.unix_timestamp("request_datetime")
            ) / 60.0
        )
    )

    # Flags de calidad
    .withColumn(
        "flag_invalid_miles",
        F.col("trip_miles") <= 0
    )

    .withColumn(
        "flag_invalid_trip_time",
        F.col("trip_time") <= 0
    )

    .withColumn(
        "flag_pickup_after_dropoff",
        F.col("pickup_datetime") > F.col("dropoff_datetime")
    )

    .withColumn(
        "flag_negative_fare",
        F.col("base_passenger_fare") < 0
    )

    .withColumn(
        "flag_negative_driver_pay",
        F.col("driver_pay") < 0
    )

    .withColumn(
        "flag_request_after_pickup",
        F.col("request_datetime") > F.col("pickup_datetime")
    )

    .withColumn(
        "flag_severe_request_anomaly",
        F.when(
            F.col("request_datetime") > F.col("pickup_datetime"),
            (
                F.unix_timestamp("request_datetime") -
                F.unix_timestamp("pickup_datetime")
            ) / 60.0 > 60
        ).otherwise(False)
    )
)

print("Reglas de calidad creadas.")

In [ ]:
quality_summary = df_quality.agg(

    F.count("*").alias("raw_records"),

    F.sum(
        F.when(F.col("flag_invalid_miles"), 1).otherwise(0)
    ).alias("invalid_trip_miles"),

    F.sum(
        F.when(F.col("flag_invalid_trip_time"), 1).otherwise(0)
    ).alias("invalid_trip_time"),

    F.sum(
        F.when(F.col("flag_pickup_after_dropoff"), 1).otherwise(0)
    ).alias("pickup_after_dropoff"),

    F.sum(
        F.when(F.col("flag_negative_fare"), 1).otherwise(0)
    ).alias("negative_fare"),

    F.sum(
        F.when(F.col("flag_negative_driver_pay"), 1).otherwise(0)
    ).alias("negative_driver_pay"),

    F.sum(
        F.when(F.col("flag_request_after_pickup"), 1).otherwise(0)
    ).alias("request_after_pickup"),

    F.sum(
        F.when(F.col("flag_severe_request_anomaly"), 1).otherwise(0)
    ).alias("severe_request_anomaly")
)

quality_summary.show(truncate=False)

print("=== VALORES DE REFERENCIA DEL PROYECTO ===")
print("trip_miles <= 0          : 95,000")
print("trip_time <= 0           : 4,006")
print("negative fare            : 97,110")
print("negative driver pay      : 4,038")
print("pickup > dropoff         : 30,340")
print("request > pickup         : 7,820,970")
print("request anomaly > 60 min : 209")

In [ ]:
# ============================================================
# FILTROS DUROS
# ============================================================
# Sólo se eliminan registros que rompen la consistencia básica
# del viaje.
#
# NO se eliminan automáticamente:
# - tarifas negativas
# - driver pay negativo
# - request_datetime > pickup_datetime
#
# Esos casos se preservan mediante flags.
# ============================================================

df_curated = df_quality.filter(
    (F.col("trip_miles") > 0) &
    (F.col("trip_time") > 0) &
    (F.col("pickup_datetime") <= F.col("dropoff_datetime"))
)

raw_count = df.count()
curated_count = df_curated.count()

removed = raw_count - curated_count
removed_pct = (removed / raw_count) * 100

print("=== RESULTADO DE CALIDAD ===")
print(f"Registros RAW      : {raw_count:,}")
print(f"Registros CURATED  : {curated_count:,}")
print(f"Registros removidos: {removed:,}")
print(f"Porcentaje removido: {removed_pct:.4f}%")

print("\n=== RESULTADOS ESPERADOS DEL PROYECTO ===")
print("RAW      : 710,576,363")
print("CURATED  : 710,451,024")
print("Removidos: 125,339")
print("Porcentaje aproximado: 0.0176%")